In [12]:
import cv2
import numpy as np
from numba import cuda
import math

@cuda.jit
def sobel_kernel(img,out):

    x,y = cuda.grid(2)

    rows,cols = img.shape

    if x>0 and y>0 and x<rows-1 and y<cols-1:

        gx = (
            -img[x-1,y-1] + img[x-1,y+1]
            -2*img[x,y-1] + 2*img[x,y+1]
            -img[x+1,y-1] + img[x+1,y+1]
        )

        gy = (
            -img[x-1,y-1]
            -2*img[x-1,y]
            -img[x-1,y+1]
            +img[x+1,y-1]
            +2*img[x+1,y]
            +img[x+1,y+1]
        )

        mag = math.sqrt(gx*gx + gy*gy)

        if mag > 255:
            mag = 255

        out[x,y] = mag


img = cv2.imread("/content/12003.jpg",0)

output = np.zeros_like(img)

d_img = cuda.to_device(img)
d_out = cuda.device_array_like(output)

threads = (16,16)

blocks = (
    math.ceil(img.shape[0]/16),
    math.ceil(img.shape[1]/16)
)

sobel_kernel[blocks,threads](d_img,d_out)

output = d_out.copy_to_host()

cv2.imwrite("sobel_output.jpg",output)

True